[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/crewai/crewai_orchestrator_workers_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
        <br>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">使用 Arize Phoenix 追踪 CrewAI - 编排器与工作智能体工作流</h1>

In [ ]:
%pip install -q arize-phoenix opentelemetry-sdk opentelemetry-exporter-otlp crewai crewai_tools openinference-instrumentation-crewai

# 设置密钥与依赖

注意：运行此 Colab 需要：

*   OpenAI API 密钥 (https://openai.com/)
*   Serper API 密钥 (https://serper.dev/)
*   Phoenix API 密钥 (https://app.phoenix.arize.com/)

In [ ]:
import os
from getpass import getpass

# 尚未设置 API 密钥时提示用户输入
openai_key = os.getenv("OPENAI_API_KEY", "OPENAI_API_KEY")
serper_key = os.getenv("SERPER_API_KEY", "SERPER_API_KEY")

if openai_key == "OPENAI_API_KEY":
    openai_key = getpass("请输入 OPENAI_API_KEY： ")

if serper_key == "SERPER_API_KEY":
    serper_key = getpass("请输入 SERPER_API_KEY： ")

# 使用提供的密钥设置环境变量
os.environ["OPENAI_API_KEY"] = openai_key
os.environ["SERPER_API_KEY"] = serper_key

In [ ]:
if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

## 配置追踪

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="crewai-agents")

# 为 CrewAI 接入插桩

In [ ]:
from openinference.instrumentation.crewai import CrewAIInstrumentor

CrewAIInstrumentor().instrument(skip_dep_check=True, tracer_provider=tracer_provider)

## 定义智能体

In [ ]:
from crewai import Agent, Crew, Task

# 定义工作智能体
trend_researcher = Agent(
    role="AI Trend Researcher",
    goal="分析 AI 当前进展",
    backstory="擅长跟踪和分析人工智能新趋势。",
    verbose=True,
)

policy_analyst = Agent(
    role="AI Policy Analyst",
    goal="研究 AI 监管与治理的影响",
    backstory="跟踪各国政府和组织的 AI 政策进展。",
    verbose=True,
)

risk_specialist = Agent(
    role="AI Risk Specialist",
    goal="识别前沿 AI 开发的潜在风险",
    backstory="专注先进 AI 的安全、对齐与滥用风险。",
    verbose=True,
)

synthesizer = Agent(
    role="Synthesis Writer",
    goal="将所有发现总结为连贯的最终报告",
    backstory="擅长将研究洞察汇编为面向管理层的叙述。",
    verbose=True,
)

orchestrator = Agent(
    role="Orchestrator",
    goal=(
        "你的任务是使用 'Delegate work to coworker' 工具，将研究与写作任务分配给正确的同事。\n"
        "每次分配任务时，必须使用以下 JSON 输入调用工具：\n\n"
        "{\n"
        '  "task": "要执行任务的简短摘要（纯字符串）",\n'
        '  "context": "任务的重要性或与报告的关系（纯字符串）",\n'
        '  "coworker": "以下原始名称之一：AI Trend Researcher, AI Policy Analyst, AI Risk Specialist, Synthesis Writer"\n'
        "}\n\n"
        "重要：\n"
        "- 不要将 'task' 或 'context' 格式化为字典。\n"
        "- 不要包含类型或嵌套描述。\n"
        "- 两者都只能使用纯字符串。\n"
        "- 多次调用工具，每位同事一次。"
    ),
    backstory="你负责将 AI 报告的各部分分配给正确的专家。",
    verbose=True,
    allow_delegation=True,
)

## 定义任务

In [ ]:
# 仅为编排器定义初始任务
initial_task = Task(
    description="创建 AI 趋势报告，包括近期创新、政策更新与安全风险，再综合为统一摘要。",
    expected_output="通过 DelegateWorkTool 分配子任务，并返回最终报告。",
    agent=orchestrator,
)

## 创建 Crew

In [ ]:
crew = Crew(
    agents=[trend_researcher, policy_analyst, risk_specialist, synthesizer],
    tasks=[initial_task],
    manager_agent=orchestrator,
    verbose=True,
)
# 运行完整工作流
result = crew.kickoff()
print(result)

### 在 Phoenix 项目中查看运行产生的追踪与跨度。